In [87]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [88]:
#load dataset

df = pd.read_csv('Churn_Modelling.csv')
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [89]:
# preprocess the data

df.drop(columns =['RowNumber', 'CustomerId', 'Surname'], inplace = True)

In [90]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [91]:
# label encoding for gender
lr_gender = LabelEncoder()
df['Gender'] = lr_gender.fit_transform(df['Gender'])

# one hot encoding for geography
ohe_geo = OneHotEncoder(sparse_output = False)
geo_encoded = ohe_geo.fit_transform(df[['Geography']])

geo_encoded_df = pd.DataFrame(geo_encoded, columns = ohe_geo.get_feature_names_out(['Geography']))
df = pd.concat([df.drop('Geography', axis = 1), geo_encoded_df], axis = 1)



In [92]:
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [93]:
# separate input and output feature

x = df.drop('Exited', axis = 1)
y = df['Exited']

In [94]:
# train test split

x_train, x_test, y_train, y_test = train_test_split(x,  y, test_size= 0.2, random_state = 42)

In [95]:
scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [96]:
with open('lr_gender.pkl','wb') as file:
    pickle.dump(lr_gender,file)

with open('ohe_geo.pkl', 'wb') as file:
    pickle.dump(ohe_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

# ANN implementation

In [97]:
import tensorflow
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense,Dropout,BatchNormalization
from keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [98]:
model = Sequential()
model.add(Dense(64, activation = 'relu', input_shape = (x_train.shape[1],)))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(Dense(32, activation ='relu'))
model.add(Dropout(0.2))
model.add(Dense(1, activation = 'sigmoid'))

model.summary()

/Users/rachin/Desktop/rachin/ai/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_23 (Dense)                │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,201 (12.50 KB)

 Trainable params: 3,073 (12.00 KB)

 Non-trainable params: 128 (512.00 B)

In [99]:
# compile the model

adam = keras.optimizers.Adam(learning_rate =0.01)
model.compile(loss = 'binary_crossentropy', optimizer = adam, metrics = ['accuracy'])

In [100]:
# set up the tensorboard
log_dir = 'log/fit'
tensorflow_callback = TensorBoard(log_dir = log_dir, histogram_freq = 1 )

In [101]:
# early stopping to prevent overfitting

callback = EarlyStopping(
    monitor = "val_loss",
    min_delta = 0.00001,
    patience = 10,
    verbose = 1,
    mode = "auto",
    baseline = None,
    restore_best_weights = True
)

In [102]:
# training the model 

history = model.fit(x_train, y_train, epochs = 100, validation_data =(x_test, y_test), callbacks = [callback, tensorflow_callback])

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8155 - loss: 0.4330 - val_accuracy: 0.8575 - val_loss: 0.3644
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8404 - loss: 0.3822 - val_accuracy: 0.8580 - val_loss: 0.3526
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8447 - loss: 0.3752 - val_accuracy: 0.8605 - val_loss: 0.3498
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8512 - loss: 0.3626 - val_accuracy: 0.8580 - val_loss: 0.3453
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8481 - loss: 0.3636 - val_accuracy: 0.8625 - val_loss: 0.3558
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8432 - loss: 0.3661 - val_accuracy: 0.8655 - val_loss: 0.3476
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8503 - loss: 0.3618 - val_accuracy: 0.8615 - val_loss: 0.3439
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8481 - loss: 0.3638 - val_accu

In [103]:
model.save('model.h5')

In [104]:
#load tensorboard extention

%load_ext tensorboard



The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [105]:

%tensorboard --logdir log/fit

Reusing TensorBoard on port 6006 (pid 12977), started 0:45:25 ago. (Use '!kill 12977' to kill it.)

In [106]:
from sklearn.metrics import accuracy_score

In [107]:
y_pred = model.predict(x_test)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  


In [108]:
y_pred = y_pred.argmax(axis = 1)

In [109]:
accuracy= accuracy_score(y_test, y_pred)
accuracy

0.8035